# MLPilot 2.3: the autonomous engine and enterprise operations

This notebook runs every 2.3 feature on the Titanic passenger list, one section at a time. Each
section shows the Python call and, where there is one, the matching `mlpilot` command.

| | Section | What you'll see |
|---|---|---|
| 1 | Autonomous engine | a goal written in English becomes a gated, resumable workflow that stops at a fairness gate, waits for approval and resumes |
| 2 | Data observability | freshness, volume, schema, null and distribution checks against SLAs; reconciliation of a copy |
| 3 | Incidents | root-cause diagnosis, a playbook, verified recovery |
| 4 | Backups & DR | a checksummed backup and a disaster-recovery drill with measured RTO and RPO |
| 5 | Supply chain | scanning model files (including a malicious one) and an SBOM |
| 6 | Serving limits | rate limiting and request validation on the REST API |
| 7 | Model governance | risk levels, approvals, retirement |
| 8 | Active learning | choosing what to label and tracking the labels |
| 9 | Privacy attacks | membership inference and model extraction against our own model |
| 10 | Hub tenants | organisations, quotas, isolation and usage metering (needs `[server]`) |

**Install:** `pip install "dataspoof-mlpilot[server]" matplotlib`. Section 10 is skipped without
`[server]`. The notebook downloads the Titanic CSV (61 KB) and runs in about 5 minutes on a
laptop. Everything it writes goes to `enterprise_outputs/`.

In [1]:
import json
import os
import shutil
import subprocess
import sys
import urllib.request
import warnings
from pathlib import Path

import polars as pl
from IPython.display import Markdown, display

import mlpilot as ac
from mlpilot import orchestration as orc

warnings.filterwarnings("ignore")
pl.Config.set_tbl_rows(10)
os.environ["AC_USER"] = "alice"          # who is acting: recorded in the registry and run history (CI sets its own)
OUT = Path("enterprise_outputs")
shutil.rmtree(OUT, ignore_errors=True)
(OUT / "data").mkdir(parents=True)
DATA = OUT / "data" / "titanic.csv"
urllib.request.urlretrieve("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv", DATA)
REGISTRY = OUT / "registry"

def mlpilot_cli(*args):
    """Run an `mlpilot` command (the same as typing it in a terminal) and show its output."""
    res = subprocess.run([sys.executable, "-m", "mlpilot.deploy", *map(str, args)], capture_output=True, text=True)
    print((res.stdout + res.stderr).strip())
    return res.returncode

from importlib.metadata import version
print("MLPilot", version("dataspoof-mlpilot"), "|", pl.read_csv(DATA).shape, "Titanic rows x columns")

MLPilot 2.3.0 | (891, 12) Titanic rows x columns


## 1 · The autonomous engine

**Write the goal in plain English.** Deterministic rules read the target, the data file, metric
floors, latency limits, fairness columns, time budget and deployment stage. With
`ANTHROPIC_API_KEY` set, Claude can draft the rest. Either way the goal is validated like a
hand-written YAML file. The LLM never executes anything.

In [2]:
goal = orc.goal_from_text(
    f"Predict Survived from {DATA.as_posix()} with AUC above 0.80 and latency below 200 ms; "
    "check fairness across Sex; deploy to production within 60 seconds",
    use_llm=False, name="titanic", registry=str(REGISTRY), include=["logistic_regression", "random_forest"])
print(goal.to_yaml())

name: titanic
data: enterprise_outputs/data/titanic.csv
target: Survived
task: auto
metric: roc_auc
min_metrics:
  roc_auc: 0.8
max_metrics: {}
time_budget: 60.0
include:
- logistic_regression
- random_forest
max_latency_ms: 200.0
max_model_mb: null
budget_usd: null
cpu_hour_usd: 0.05
test_size: 0.2
group: null
time: null
contract: null
min_quality: 50.0
max_suspicious_share: null
sensitive:
- Sex
fairness_limits: null
robustness: true
max_membership_auc: null
explain: true
deploy: production
approvals:
  production: 1
registry: enterprise_outputs\registry
monitor: true
seed: 0
description: Predict Survived from enterprise_outputs/data/titanic.csv with AUC above
  0.80 and latency below 200 ms; check fairness across Sex; deploy to production within
  60 seconds



In [3]:
print(orc.plan(goal).describe())      # only registered tools; promotion needs an approval

Plan for 'titanic': predict 'Survived'
   1. ingest               Read the goal's data source (or the DataFrame passed to the run) and check the target exists
   2. quality_gate         Score data quality (0-100) and validate the data contract; stop below the goal's floor
   3. split                Leak-checked train/test split (stratified, grouped or by time)
   4. poisoning_check      Look for label errors, conflicting duplicates and backdoor triggers in the training rows
   5. automl               Train models with AutoML within the goal's time budget; with latency or size limits, search for the best model that meets them (cost-aware)
   6. evaluate             Held-out metrics with confidence intervals; stop if a metric misses the goal
   7. performance_profile  Measure single-row latency (p50/p95), batch throughput and model size; stop if a limit in the goal is exceeded
   8. explain              SHAP global importance of the features
   9. fairness             Bias checks across 

**Run it.** The engine executes the plan step by step and stops at the first failed gate:

In [4]:
RUN = OUT / "runs" / "titanic"
res = orc.run(goal, RUN)                 # requested by AC_USER (alice)
print(res.summary())

Run titanic: FAILED  (71s, 88 CPU-s, $0.0012)
  [ok ] ingest               done
  [ok ] quality_gate         done
  [ok ] split                done
  [ok ] poisoning_check      done
  [ok ] automl               done
  [ok ] evaluate             done
  [ok ] performance_profile  done
  [ok ] explain              done
  [ERR] fairness             failed: gate: Fairness limits broken: demographic_parity_difference 0.631 > 0.1: gap in the share of each group predicted positive; disparate_impact_ratio 0.212 < 0.8: 
  [-- ] robustness           skipped: stopped after fairness failed
  [-- ] artifact_scan        skipped: stopped after fairness failed
  [-- ] model_card           skipped: stopped after fairness failed
  [-- ] register             skipped: stopped after fairness failed
  [-- ] monitor_setup        skipped: stopped after fairness failed
  [-- ] promote              skipped: stopped after fairness failed


The **fairness gate** stopped the run, and that is the correct decision for this data. On the
Titanic, "women and children first" makes `Sex` a real driver of survival, so the model treats
the groups very differently. The same gate would block a loan or hiring model with such a gap.

The steps after the gate were skipped, so nothing was registered or deployed. The run directory
keeps everything done so far.

Here we accept that `Sex` is a legitimate feature for this historical analysis, and drop the
fairness requirement. Re-running the same directory **reuses every step whose inputs didn't
change**. Only the steps that depend on the edited part of the goal run again:

In [5]:
goal = orc.Goal.from_dict({**goal.to_dict(), "sensitive": []})
res = orc.run(goal, RUN)                 # requested by AC_USER (alice)
print(res.summary())

Run titanic: WAITING_APPROVAL  (89s, 109 CPU-s, $0.0015)
  [ok ] ingest               done
  [ok ] quality_gate         done (reused)
  [ok ] split                done (reused)
  [ok ] poisoning_check      done (reused)
  [ok ] automl               done (reused)
  [ok ] evaluate             done (reused)
  [ok ] performance_profile  done (reused)
  [ok ] explain              done (reused)
  [ok ] robustness           done
  [ok ] artifact_scan        done
  [ok ] model_card           done
  [ok ] register             done
  [ok ] monitor_setup        done
  [...] promote              waiting_approval: needs 1 more approval(s) for production: mlpilot approve enterprise_outputs\runs\titanic --step promote --by <name>


Every gate passed, and the run now **waits for approval** to promote to production. Approvals
follow the four-eyes rule: the person who started the run can't approve it. Each approval is tied
to the exact inputs of the step, so changing the plan needs a new approval.

In [6]:
try:
    orc.approve(RUN, "promote", by="alice")
except PermissionError as exc:
    print("refused:", exc)
orc.approve(RUN, "promote", by="priya", note="reviewed the model card and metrics")
res = orc.resume(RUN)
print(res.summary())

refused: The person who started the run can't approve it (four-eyes rule)
Run titanic: SUCCEEDED  (89s, 109 CPU-s, $0.0015)
  [ok ] ingest               done
  [ok ] quality_gate         done (reused)
  [ok ] split                done (reused)
  [ok ] poisoning_check      done (reused)
  [ok ] automl               done (reused)
  [ok ] evaluate             done (reused)
  [ok ] performance_profile  done (reused)
  [ok ] explain              done (reused)
  [ok ] robustness           done (reused)
  [ok ] artifact_scan        done (reused)
  [ok ] model_card           done (reused)
  [ok ] register             done (reused)
  [ok ] monitor_setup        done (reused)
  [ok ] promote              done


**Everything is on record** in the run directory:
- `report.md`
- `events.jsonl`, an append-only log
- `state.json`, with timings, results and cost per step
- `manifest.json`, the environment and data fingerprint, for reproduction
- `artifacts/`: the data splits, the model, its metrics, the model card and the monitor baseline

In [7]:
display(Markdown((RUN / "report.md").read_text(encoding="utf-8")))
manifest = json.loads((RUN / "manifest.json").read_text())
print({k: manifest[k] for k in ("python", "seed", "data_fingerprint", "git_commit")})
print(len((RUN / "events.jsonl").read_text().splitlines()), "events logged;",
      sorted(p.name for p in (RUN / "artifacts").iterdir()))

# Run `titanic`: succeeded

**Goal:** Predict Survived from enterprise_outputs/data/titanic.csv with AUC above 0.80 and latency below 200 ms; check fairness across Sex; deploy to production within 60 seconds  
**Model name:** `titanic` · **target:** `Survived` · **requested by:** alice  
**Cost:** 89 s wall, 109 CPU-s, $0.0015 at $0.05/CPU-hour

## Requirements

- roc_auc ≥ 0.8
- p95 latency ≤ 200.0 ms

## Steps

| Step | Status | Seconds | Result |
|---|---|---|---|
| ingest | done | 0.005 | {"source": "enterprise_outputs/data/titanic.csv", "rows": 891, "columns": 12, "target_missing": 0} |
| quality_gate | done (reused) | 0.085 | {"score": 97.2, "grade": "A"} |
| split | done (reused) | 0.024 | {"strategy": "stratified by 'Survived'", "train_rows": 713, "test_rows": 178, "issues": []} |
| poisoning_check | done (reused) | 1.703 | {"suspicious_rows": 102, "suspicious_share": 0.1431} |
| automl | done (reused) | 55.389 | {"best": "ensemble", "cv_score": 0.8941872486621703, "metric": "roc_auc", "trials": 60, "attempts": [{"attempt": 1, "best": "ensemble", "cv_score": 0.8942, "ensemble": true, "excluded": [], "latency_ms_p50": 41.12, "latency_ms_p95": 51.01, "model_mb": 6.321, "meets_limits": true}], "leaderboard": [{ |
| evaluate | done (reused) | 1.136 | {"metrics": {"accuracy": 0.8427, "balanced_accuracy": 0.8334, "precision_macro": 0.8334, "recall_macro": 0.8334, "f1_macro": 0.8334, "f1_weighted": 0.8427, "mcc": 0.6668, "cohen_kappa": 0.6668, "precision": 0.7941, "recall": 0.7941, "f1": 0.7941, "specificity": 0.8727, "roc_auc": 0.8854, "pr_auc": 0 |
| performance_profile | done (reused) | 1.71 | {"latency_ms_p50": 46.22, "latency_ms_p95": 62.49, "model_mb": 6.321, "throughput_rows_per_s": 3120} |
| explain | done (reused) | 10.407 | {"top_features": [{"feature": "Sex_female", "mean_abs_shap": 0.11998004787158655, "share": 0.26179649317779957}, {"feature": "Sex_male", "mean_abs_shap": 0.11213485535786356, "share": 0.24467828123479707}, {"feature": "Pclass", "mean_abs_shap": 0.07476203541227887, "share": 0.16313077916685997}, {"f |
| robustness | done (reused) | 9.711 | {"noise_flip_5pct": 0.04666666666666667, "median_adversarial_distance": 0.1, "extreme": {"crashed": [], "non_finite": []}} |
| artifact_scan | done (reused) | 0.136 | {"files": 1, "critical": [], "warnings": []} |
| model_card | done (reused) | 7.867 | {"path": "enterprise_outputs\\runs\\titanic\\artifacts\\model_card.html"} |
| register | done (reused) | 0.154 | {"name": "titanic", "version": 1, "stage": "dev"} |
| monitor_setup | done (reused) | 0.25 | {"path": "enterprise_outputs\\runs\\titanic\\artifacts\\monitor.json"} |
| promote | done | 0.011 | {"name": "titanic", "version": 1, "stage": "production"} |

Files: `workflow.json`, `state.json`, `events.jsonl`, `approvals.json`, `manifest.json`, `artifacts/` (splits, model, metrics, model card, monitor baseline).


{'python': '3.13.14', 'seed': 0, 'data_fingerprint': 'fd36dd827206e8ad', 'git_commit': None}
62 events logged; ['data.parquet', 'metrics.json', 'model', 'model_card.html', 'monitor.json', 'quality.txt', 'registered.json', 'test.parquet', 'train.parquet']


**The same from a terminal.** Exit codes: 0 succeeded, 3 waiting for approval, 1 failed.

```bash
mlpilot plan --text "predict Survived from titanic.csv with AUC above 0.80" --save goal.yml
mlpilot run --goal goal.yml --out runs/titanic
mlpilot approve runs/titanic --step promote --by priya
mlpilot run --out runs/titanic --resume
mlpilot status runs/titanic
```

In [8]:
mlpilot_cli("status", RUN)

Run titanic: SUCCEEDED  (89s, 109 CPU-s, $0.0015)
  [ok ] ingest               done
  [ok ] quality_gate         done (reused)
  [ok ] split                done (reused)
  [ok ] poisoning_check      done (reused)
  [ok ] automl               done (reused)
  [ok ] evaluate             done (reused)
  [ok ] performance_profile  done (reused)
  [ok ] explain              done (reused)
  [ok ] robustness           done (reused)
  [ok ] artifact_scan        done (reused)
  [ok ] model_card           done (reused)
  [ok ] register             done (reused)
  [ok ] monitor_setup        done (reused)
  [ok ] promote              done


0

## 2 · Data observability & reconciliation

`DataMonitor` checks every new batch against SLAs and against its own history:
- freshness;
- volume;
- schema;
- null rates;
- distribution, as PSI against the previous batch.

The history keeps statistics, never rows. Here three normal batches build the history, then a
broken one arrives.

In [9]:
from mlpilot.quality import DataMonitor, reconcile

titanic = pl.read_csv(DATA)
mon = DataMonitor("passengers", history=OUT / "observability.jsonl",
                  sla={"min_rows": 300, "max_null_rate": {"Fare": 0.01}, "max_row_change": 0.4})
for seed in range(3):
    mon.observe(titanic.sample(600, seed=seed))

broken = (titanic.sample(150, seed=9)
          .with_columns(pl.when(pl.col("Pclass") == 3).then(None).otherwise(pl.col("Fare")).alias("Fare"),
                        (pl.col("Age") * 12).alias("Age"))           # e.g. ages arriving in months
          .drop("Embarked"))
report = mon.observe(broken)
print(report.summary())
mon.history()

passengers @ 2026-11-22T02:17:08+00:00: 150 rows - FAILED (6 issue(s))
  [error] volume: 150 rows (SLA at least 300)
  [error] volume: 150 rows vs a recent median of 600 (-75%)
  [error] schema · Embarked: column removed
  [error] nulls · Fare: 55.3% missing (SLA 1.0%)
  [warning] distribution · Age: PSI 5.59 vs the last batch
  [warning] distribution · Fare: PSI 1.59 vs the last batch


observed_at,rows,columns,staleness_hours,passed,issues
str,i64,i64,f64,bool,i64
"""2026-11-22T02:17:08+00:00""",600,12,null,true,0
"""2026-11-22T02:17:08+00:00""",600,12,null,true,0
"""2026-11-22T02:17:08+00:00""",600,12,null,true,0
"""2026-11-22T02:17:08+00:00""",150,11,null,false,6


`reconcile` proves that a copy holds exactly what the source had, after an ETL job, a migration
or a backup restore. It reports missing, extra and duplicate keys, changed rows, per-column null
counts, numeric sums and schema differences.

In [10]:
copy = titanic.filter(pl.col("PassengerId") != 7).with_columns(
    pl.when(pl.col("PassengerId") == 3).then(pl.col("Fare") + 1).otherwise(pl.col("Fare")).alias("Fare"))
print(reconcile(titanic, copy, key="PassengerId").summary())
print("identical copy passes:", reconcile(titanic, titanic.sample(fraction=1.0, shuffle=True, seed=1),
                                          key="PassengerId").passed)

Reconciliation: FAILED  source 891 rows, destination 890 rows
  key 'PassengerId': 1 missing in destination, 0 extra, duplicates {'source': 0, 'destination': 0}, 1 changed rows
  column PassengerId: source nulls 0, destination nulls 0, sums 397386.0 vs 397379.0
  column Pclass: source nulls 0, destination nulls 0, sums 2057.0 vs 2056.0
  column Age: source nulls 177, destination nulls 177, sums 21205.17 vs 21151.17
  column Fare: source nulls 0, destination nulls 0, sums 28693.9493 vs 28643.086800000005
identical copy passes: True


## 3 · Incidents: diagnose, act, verify

`diagnose` combines the model monitor, the data monitor and service metrics, and names the most
likely root cause. Each cause comes with a playbook:
- **Low-risk actions** run automatically: notify, pause retraining, scale out, start a retraining
  run.
- **High-risk actions**, such as a rollback, wait for an approver.

An incident closes only after a fresh check is healthy.

In [11]:
from mlpilot.monitoring import IncidentLog, Monitor, diagnose

model = ac.load_model(RUN / "artifacts" / "model")
test = pl.read_parquet(RUN / "artifacts" / "test.parquet")
monitor = Monitor(model, test, name="titanic")

shifted = test.drop("Survived").with_columns((pl.col("Age") + 30).alias("Age"), (pl.col("Fare") * 5).alias("Fare"),
                                             pl.lit(1, dtype=pl.Int64).alias("Pclass"))   # everyone in first class
diag = diagnose(monitor=monitor.check(shifted))
print(diag.summary())

log = IncidentLog(OUT / "incidents.jsonl", registry=REGISTRY)
incident = log.remediate(log.open(diag, model="titanic"),
                         handlers={"notify": lambda inc: "posted to #ml-alerts",
                                   "retrain": lambda inc: "retraining run queued (promotion still gated)"})
print({a.name: a.status for a in incident.actions})
incident = log.verify(incident, monitor.check(test.drop("Survived")))    # a healthy batch
print("incident", incident.id, "->", incident.status)

Diagnosis: covariate_shift (confidence medium)
  evidence: data_drift: 38% of columns drifted strongly: ['Pclass', 'Fare', 'Age']
  evidence: prediction_drift: prediction distribution shifted (PSI 2.893)
  action [low]: retrain - retrain including the recent data
  action [low]: review_features - ask the owners to review the drifted features
  action [low]: notify - alert the owners (on-call, data owner)
{'retrain': 'done', 'review_features': 'no_handler', 'notify': 'done'}


incident INC-03e8d6aa -> resolved


## 4 · Backups & disaster recovery

A backup turns each source into a checksummed archive, and SQLite databases are snapshotted
consistently even while in use. A restore verifies every checksum first and refuses unsafe
paths. A **drill** backs up, restores into a scratch area, compares every file and measures RTO
and RPO. CI runs a drill before every release.

```bash
mlpilot backup --source registry=registry --source runs=runs --out backups --keep-last 14
mlpilot dr-drill --source registry=registry --workdir dr --rto-limit 120
```

In [12]:
from mlpilot.backup import backup, dr_drill, verify_backup

m = backup({"registry": REGISTRY, "runs": OUT / "runs"}, OUT / "backups")
print(m.path.name, verify_backup(m.path))
print(dr_drill({"registry": REGISTRY}, OUT / "dr", rto_limit=60).summary())

backup-20261122T021710156544Z {'registry': 'ok', 'runs': 'ok'}


DR drill: PASSED - RTO 0.06s, RPO 2.21s
  registry: 5 files, 0 mismatched, 0 missing


## 5 · Supply-chain security

Pickle and joblib files can run code when loaded. `scan_model` replays them with a restricted
unpickler, so nothing in the file executes. It flags anything that would run code. Our model is
clean; a deliberately malicious file is not. The engine runs the same scan before it registers a
model.

In [13]:
import os
import pickle
from mlpilot.supply_chain import generate_sbom, scan_model

print(scan_model(RUN / "artifacts" / "model").summary())

class Payload:                                        # what an attacker hides in a "model.pkl"
    def __reduce__(self):
        return (os.system, ("echo you have been pwned",))
(OUT / "suspicious").mkdir()
(OUT / "suspicious" / "model.pkl").write_bytes(pickle.dumps({"weights": [0.1, 0.2], "x": Payload()}))
print(scan_model(OUT / "suspicious").summary())       # nothing was executed to find this

sbom = generate_sbom(OUT / "sbom.cdx.json")
print(len(sbom["components"]), "components in the SBOM, e.g.",
      [c["purl"] for c in sbom["components"][:3]])

Model scan of enterprise_outputs\runs\titanic\artifacts\model: PASSED - 1 file(s), 0 critical, 0 warning(s)
Model scan of enterprise_outputs\suspicious: FAILED - 1 file(s), 1 critical, 0 warning(s)
  [critical] model.pkl: global nt.system


21 components in the SBOM, e.g. ['pkg:pypi/cloudpickle@3.1.3', 'pkg:pypi/dataspoof-mlpilot@2.3.0', 'pkg:pypi/dnspython@2.9.2']


## 6 · Serving limits

The REST API can rate-limit each client and validate requests against the columns the model was
trained on:
- **Rate limits** use a token bucket per API key or address. Requests over the limit get `429`
  with `Retry-After`.
- **Validation:** `strict` rejects bad records with `422`; `warn` scores them and lists the
  problems.

In production, set `AC_RATE_LIMIT=120/minute` and `AC_VALIDATE=strict`.

In [14]:
import io

app = ac.make_app(model, rate_limit="3/minute", validate="strict")

def post(records):
    body = json.dumps({"records": records}).encode()
    env = {"REQUEST_METHOD": "POST", "PATH_INFO": "/predict", "CONTENT_LENGTH": str(len(body)),
           "CONTENT_TYPE": "application/json", "wsgi.input": io.BytesIO(body), "REMOTE_ADDR": "10.1.2.3"}
    status = {}
    payload = json.loads(b"".join(app(env, lambda s, h: status.update(code=s, headers=dict(h)))))
    return status["code"], status["headers"].get("Retry-After"), payload

passenger = test.drop("Survived").head(1).to_dicts()[0]
requests = [("valid", [passenger]),
            ("Age='forty'", [{**passenger, "Age": "forty"}]),
            ("no Sex column", [{k: v for k, v in passenger.items() if k != "Sex"}]),
            ("4th this minute", [passenger])]
for label, records in requests:
    code, retry, body = post(records)
    result = body.get("predictions") or body.get("error")
    print(f"{label:<16} {code:<26} {result}" + (f"  (Retry-After {retry}s)" if retry else ""))

valid            200 OK                     [1]
Age='forty'      422 Unprocessable Entity   Invalid records: record 0: Age='forty' is not a number
no Sex column    422 Unprocessable Entity   Invalid records: record 0: missing ['Sex']
4th this minute  429 Too Many Requests      Rate limit exceeded; slow down.  (Retry-After 20s)


## 7 · Model governance: risk levels, approvals, retirement

Classify each model's risk:
- **High risk:** production needs two approvers other than the promoter, and a risk review that
  isn't overdue.
- **Unacceptable:** the model can never leave dev.
- **Retiring** a model archives every version, blocks loading and promotion, and points users to
  the replacement.

In [15]:
from mlpilot.registry import ModelRegistry, ModelRetiredError, PromotionError

reg = ModelRegistry(REGISTRY)
reg.set_risk("titanic", "high", reason="decisions about people", owner="ml-governance", review_by="2027-06-30")
v2 = reg.register(model, "titanic", description="retrained candidate")
try:
    reg.promote("titanic", v2.version, "production", approved_by=["priya"])
except PromotionError as exc:
    print("blocked:", exc)
print("promoted:", reg.promote("titanic", v2.version, "production", approved_by=["priya", "omar"]))
reg.versions("titanic").select("version", "stage")

blocked:

 titanic v2 can't move to production: high-risk model: production needs two approvers other than alice (approved_by=['priya'])
promoted: ModelVersion(titanic v2, production)


version,stage
i64,str
1,"""archived"""
2,"""production"""


## 8 · Active learning & labeling

When labels are expensive, label the rows the model is least sure about, spread across the data.
The `LabelQueue` tracks who labelled what, takes the majority vote and measures how much
labelers agree. `simulate_active_learning` replays the process on data you already have
labels for, to check that it beats random labeling before you pay for labels.

In [16]:
from mlpilot.labeling import LabelQueue, select_for_labeling, simulate_active_learning

pool = test.drop("Survived")
picks = select_for_labeling(model, pool, n=8, id_col="PassengerId")
print(picks.select("PassengerId", "Sex", "Age", "Pclass", "uncertainty", "predicted"))

queue = LabelQueue(OUT / "labels")
queue.add(picks, id_col="PassengerId")
truth = dict(zip(test["PassengerId"].cast(pl.Utf8), test["Survived"].cast(pl.Utf8)))
for item in queue.items()["item_id"].to_list()[:5]:
    queue.record(item, truth[item], by="asha")
    queue.record(item, truth[item] if item != queue.items()["item_id"][0] else "0", by="ben")
print("pending:", queue.pending().height, "| agreement:", queue.agreement())

curve = simulate_active_learning(titanic.drop("PassengerId"), "Survived", initial=30, batch=20, rounds=6)
curve.pivot(on="strategy", index="labelled", values="accuracy")

shape: (8, 6)
┌─────────────┬────────┬──────┬────────┬─────────────┬───────────┐
│ PassengerId ┆ Sex    ┆ Age  ┆ Pclass ┆ uncertainty ┆ predicted │
│ ---         ┆ ---    ┆ ---  ┆ ---    ┆ ---         ┆ ---       │
│ i64         ┆ str    ┆ f64  ┆ i64    ┆ f64         ┆ str       │
╞═════════════╪════════╪══════╪════════╪═════════════╪═══════════╡
│ 828         ┆ male   ┆ 1.0  ┆ 2      ┆ 0.994926    ┆ 1         │
│ 262         ┆ male   ┆ 3.0  ┆ 3      ┆ 0.986443    ┆ 0         │
│ 246         ┆ male   ┆ 44.0 ┆ 1      ┆ 0.986133    ┆ 0         │
│ 621         ┆ male   ┆ 27.0 ┆ 3      ┆ 0.971432    ┆ 0         │
│ 680         ┆ male   ┆ 36.0 ┆ 1      ┆ 0.963033    ┆ 1         │
│ 485         ┆ male   ┆ 25.0 ┆ 1      ┆ 0.960153    ┆ 1         │
│ 432         ┆ female ┆ null ┆ 3      ┆ 0.917474    ┆ 0         │
│ 299         ┆ male   ┆ null ┆ 1      ┆ 0.823036    ┆ 1         │
└─────────────┴────────┴──────┴────────┴─────────────┴───────────┘


pending: 3 | agreement: {'items': 5, 'mean_agreement': 0.9, 'unanimous': 0.8}


labelled,margin,random
i64,f64,f64
30,0.7004,0.7004
50,0.7266,0.618
70,0.7079,0.6854
90,0.7191,0.7004
110,0.7079,0.6929
130,0.7004,0.7004
150,0.6966,0.6966


On the Titanic, the simulation uses only a handful of numeric columns, and the two strategies
end up close. That is exactly why the simulation exists: check on your own labelled data before
paying for a labeling programme. Uncertainty sampling pays off most when there are many features
and the classes overlap.

## 9 · Privacy attacks against our own model

- **Membership inference:** can someone tell whether a passenger was in the training data? An
  attack AUC of 0.5 means nothing leaks.
- **Extraction:** how many API queries does it take to clone the model? The engine can gate a
  release on the first test (`max_membership_auc`).

In [17]:
from mlpilot.model_privacy import extraction_test, membership_inference_test

train = pl.read_parquet(RUN / "artifacts" / "train.parquet")
print(membership_inference_test(model, train, test).summary())
print(extraction_test(model, test, queries=(50, 500, 2000)).summary())

Membership inference: attack AUC 0.590, advantage 0.163 -> medium risk (true-label confidence: train 0.776, held-out 0.709)


Model extraction (decision-tree copy agreeing with your model on real rows): 50 queries -> 75%, 500 queries -> 80%, 2000 queries -> 85%


How to read these results:
- **Membership inference:** the random forest is more confident on passengers it trained on, so
  an attacker does better than chance (medium risk). A regularised model, or more training data,
  lowers this. Set `max_membership_auc` in the goal to make it a release gate.
- **Extraction:** a simple copy agrees with the model on most passengers after a few thousand
  queries. Rate limits (section 6), returning labels rather than probabilities, and query
  monitoring make that harder.

## 10 · Hub tenants (needs `pip install "dataspoof-mlpilot[server]"`)

The Hub is MLPilot's multi-user server. Organisations (tenants) own workspaces and users, with
quotas on workspaces, model versions, storage and runs per day, and every request is metered.
This section starts a throwaway Hub in-process to show it; in production it runs with
`mlpilot hub serve`.

In [18]:
try:
    from fastapi.testclient import TestClient
    from mlpilot.hub.app import bootstrap_admin, create_app
except ImportError:
    print("Hub not installed - skipping (pip install 'dataspoof-mlpilot[server]')")
else:
    url = f"sqlite:///{(OUT / 'hub.db').resolve().as_posix()}"
    admin = {"Authorization": f"Bearer {bootstrap_admin(url, name='root', email='root@platform')}"}
    hub = TestClient(create_app(database_url=url, artifact_root=str(OUT / "hub-artifacts")))
    hub.post("/api/orgs", json={"name": "acme", "quotas": {"max_workspaces": 1, "max_runs_per_day": 2}}, headers=admin)
    hub.post("/api/orgs", json={"name": "globex"}, headers=admin)
    ana = hub.post("/api/service-accounts", json={"name": "ana"}, headers=admin).json()
    gus = hub.post("/api/service-accounts", json={"name": "gus"}, headers=admin).json()
    hub.post("/api/orgs/acme/users", json={"user": ana["id"]}, headers=admin)
    hub.post("/api/orgs/globex/users", json={"user": gus["id"]}, headers=admin)
    hub.post("/api/workspaces", json={"name": "acme-ml", "org": "acme"}, headers=admin)
    hub.put("/api/workspaces/acme-ml/members", json={"user": ana["id"], "role": "data_scientist"}, headers=admin)
    print("2nd acme workspace:", hub.post("/api/workspaces", json={"name": "acme-2", "org": "acme"},
                                          headers=admin).json()["detail"])
    print("add globex user to acme:", hub.put("/api/workspaces/acme-ml/members",
                                              json={"user": gus["id"], "role": "viewer"}, headers=admin).json()["detail"])
    ana_h = {"Authorization": f"Bearer {ana['token']}"}
    codes = [hub.post("/api/workspaces/acme-ml/runs", json={"experiment": "e", "name": f"run {i}"},
                      headers=ana_h).status_code for i in range(3)]
    print("ana's runs today:", codes, "(the third is over the quota)")
    print("usage:", hub.get("/api/orgs/acme/usage", headers=ana_h).json()["totals"])

2nd acme workspace: Quota exceeded for organisation 'acme': max_workspaces = 1 (in use 1, this request adds 1)
add globex user to acme: Cross-tenant membership refused: the user belongs to another organisation (assign them with POST /api/orgs/<org>/users)
ana's runs today: [201, 201, 429] (the third is over the quota)
usage: {'api_calls': 3.0, 'runs': 2.0}


## Retire the model, and where to go next

When a model is replaced for good, retire it so nobody loads it by accident:

In [19]:
reg.retire("titanic", reason="replaced by the 2027 passenger model", replacement="titanic-2027")
try:
    reg.load("titanic")
except ModelRetiredError as exc:
    print("refused:", exc)

refused: "titanic was retired on 2026-11-22T02:17:19+00:00 (replaced by the 2027 passenger model); use 'titanic-2027' instead"


- **Your own data:** write a goal (`mlpilot plan --text "..." --save goal.yml`), review it, then
  run it with `mlpilot run --goal goal.yml --out runs/<name>`.
- **In CI:** the generated pipelines (`mlpilot cicd init`) and the `certify` job show how to gate
  releases on these checks.
- **Guides:**
  - [Autonomous engine & operations](../docs/guide/enterprise-operations.md)
  - [Enterprise gap audit](../docs/enterprise-gap-audit.md): what is built and what is still
    missing
  - [Titanic walkthrough](titanic_walkthrough.ipynb): the individual modules step by step

Everything this notebook wrote is in `enterprise_outputs/` and can be deleted.